# euroloto — Exemples d'utilisation

Ce notebook illustre les principales fonctionnalités du package `euroloto` :
initialisation, analyse statistique, prédiction et visualisation.

---

## 0. Import et initialisation

`euroloto.init()` lit le fichier `config.yaml` situé à la racine du projet
et charge les deux bases de données XLSM en mémoire.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%matplotlib inline

import euroloto

# Chemin vers config.yaml (relatif à ce notebook)
euroloto.init('../config.yaml')

---
## 1. Données brutes

In [ ]:
# DataFrame complet — Loto
df_loto = euroloto.draws('loto')
print(f"{len(df_loto)} tirages Loto")
df_loto.tail(5)

In [ ]:
# DataFrame complet — EuroMillions
df_euro = euroloto.draws('euro')
print(f"{len(df_euro)} tirages EuroMillions")
df_euro.tail(5)

---
## 2. Fréquences

In [ ]:
# Top 10 boules les plus sorties au Loto
euroloto.frequency('loto').nlargest(10).rename('apparitions')

In [ ]:
# Graphique — boules principales Loto
# Rouge = chaud (> moyenne +10 %), Bleu = froid, Gris = neutre
euroloto.plot_frequency('loto')

In [ ]:
# Graphique — numéros chance Loto
euroloto.plot_frequency('loto', label='bonus')

In [ ]:
# Graphique — boules EuroMillions
euroloto.plot_frequency('euro')

---
## 3. Numéros chauds / froids

Comparaison de la fréquence des 50 derniers tirages par rapport à la moyenne globale.

In [ ]:
hc = euroloto.hot_cold('loto', n_recent=50)
print("5 numéros les plus CHAUDS :")
print(hc[hc['statut'] == 'chaud'].head(5)[['total_pct', 'recent_pct', 'delta']].to_string())
print("\n5 numéros les plus FROIDS :")
print(hc[hc['statut'] == 'froid'].tail(5)[['total_pct', 'recent_pct', 'delta']].to_string())

In [ ]:
euroloto.plot_hot_cold('loto', n_recent=50)

---
## 4. Co-occurrences

Combien de fois deux numéros sont sortis ensemble dans le même tirage.

In [ ]:
# Top 10 paires les plus co-occurentes
euroloto.top_pairs('loto', n=10)

In [ ]:
# Heatmap — seuil min_cooc=30 pour ne garder que les paires significatives
euroloto.plot_cooccurrence('loto', top_n=20, min_cooc=30)

In [ ]:
# Sans seuil — toutes les paires du top 15
euroloto.plot_cooccurrence('euro', top_n=15)

---
## 5. Tests statistiques d'uniformité

Vérifient si la distribution observée est compatible avec une loi uniforme
(hypothèse attendue si le jeu est équitable).

In [ ]:
import pandas as pd

rows = []
for kind in ('loto', 'euro'):
    chi2 = euroloto.chi2_test(kind)
    ks   = euroloto.ks_test(kind)
    rows.append({
        'jeu': kind,
        'chi2_stat': chi2['statistic'], 'chi2_p': chi2['pvalue'], 'chi2_uniforme': chi2['is_uniform'],
        'ks_stat':   ks['statistic'],   'ks_p':   ks['pvalue'],   'ks_uniforme':   ks['is_uniform'],
    })
pd.DataFrame(rows).set_index('jeu')

---
## 6. Statistiques sur la somme des boules

In [ ]:
print("Loto")
print(euroloto.sum_stats('loto').to_string())
print("\nEuroMillions")
print(euroloto.sum_stats('euro').to_string())

In [ ]:
euroloto.plot_sum('loto')

---
## 7. Distribution pair / impair

In [ ]:
print("Loto — distribution pair/impair par tirage :")
euroloto.even_odd('loto')

---
## 8. Retard (numéros en attente)

Nombre de tirages depuis la dernière apparition de chaque numéro.

In [ ]:
print("Top 10 numéros les plus en retard — Loto :")
euroloto.overdue('loto').head(10)

In [ ]:
euroloto.plot_overdue('euro')

---
## 9. Tendance temporelle

Fréquence normalisée (% par tirage) des 10 numéros les plus fréquents, année par année.
La normalisation corrige le saut dû au passage d'EuroMillions à 2 tirages/semaine en 2011.

In [ ]:
euroloto.plot_temporal('euro', top_n=10)

In [ ]:
euroloto.plot_temporal('loto', top_n=8)

---
## 10. Écarts entre apparitions (gaps)

In [ ]:
euroloto.plot_gaps('loto')

---
## 11. Compléments pour des numéros fixés

**Question :** si je joue toujours les numéros 26 et 27, quels sont les 3 meilleurs compléments
selon l'historique ?

In [ ]:
FIXED = [26, 27]

# Analyse sur le Loto uniquement
print(f"Loto — tirages contenant {FIXED} :")
euroloto.companions(FIXED, kind='loto')

In [ ]:
euroloto.plot_companions(FIXED, kind='loto')

In [ ]:
# Analyse croisée Loto + EuroMillions (plage commune 1–49)
print(f"Loto + EuroMillions — compléments pour {FIXED} :")
euroloto.companions(FIXED, kind='all', n_top=15)

In [ ]:
euroloto.plot_companions(FIXED, kind='all')

In [ ]:
# Exemple avec un seul numéro fixé
euroloto.companions([13], kind='loto', n_top=10)

---
## 12. Prédiction

Génération de combinaisons candidates par Monte Carlo pondéré.

**`alpha`** contrôle l'équilibre entre fréquence et retard :
- `alpha=1.0` → favorise les numéros historiquement fréquents
- `alpha=0.0` → favorise les numéros les plus en retard
- `alpha=0.6` → réglage par défaut (60 % fréquence / 40 % retard)

> **Rappel :** les tirages sont indépendants et équiprobables par construction.
> Ce modèle explore des tendances empiriques — il ne prédit pas l'avenir.

In [ ]:
# 10 combinaisons Loto libres
combos = euroloto.prediction(kind='loto', n=10, alpha=0.6, seed=42)
for i, c in enumerate(combos, 1):
    print(f"#{i:02d}  {c['main']}  +  Chance {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# 10 combinaisons EuroMillions libres
combos_euro = euroloto.prediction(kind='euro', n=10, alpha=0.5, seed=0)
for i, c in enumerate(combos_euro, 1):
    print(f"#{i:02d}  {c['main']}  +  Étoiles {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# 10 combinaisons Loto incluant obligatoirement les numéros [26, 27]
combos_fixed = euroloto.prediction(FIXED, kind='loto', n=10, alpha=0.6, seed=42)
for i, c in enumerate(combos_fixed, 1):
    main_str = '  '.join(f'[{x:02d}]' if x in FIXED else f' {x:02d} ' for x in c['main'])
    print(f"#{i:02d}  {main_str}  +  Chance {c['bonus']}   score={c['score']:.2f}")
print("  ([ ] = numéros fixés)")

In [ ]:
# Prédiction croisée Loto + EuroMillions avec numéros fixés
# Retourne un dict {'loto': [...], 'euro': [...]}
result_all = euroloto.prediction(FIXED, kind='all', n=5, alpha=0.6, seed=42)

print("=== Loto ===")
for i, c in enumerate(result_all['loto'], 1):
    print(f"  #{i}  {c['main']}  +  Chance {c['bonus']}   score={c['score']:.2f}")

print("\n=== EuroMillions ===")
for i, c in enumerate(result_all['euro'], 1):
    print(f"  #{i}  {c['main']}  +  Étoiles {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# Influence du paramètre alpha
print("alpha=1.0 (fréquence pure)")
for c in euroloto.prediction(kind='loto', n=3, alpha=1.0, seed=1):
    print(f"  {c['main']}")

print("\nalpha=0.0 (retard pur)")
for c in euroloto.prediction(kind='loto', n=3, alpha=0.0, seed=1):
    print(f"  {c['main']}")